In [ ]:
import requests
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

class WikiRAGBot:
    def __init__(self, model_id="google/flan-t5-base"):
        print(f"Loading {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

        # Check for GPU (highly recommended in Colab: Runtime > Change runtime type > T4 GPU)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        print(f"Running on {self.device}")

    def fetch_wikipedia_context(self, query):
        """Retrieves the top context from Wikipedia for the query."""
        search_url = "https://wikipedia.org"

        # 1. Search for the most relevant page
        search_params = {
            "action": "query", "list": "search", "srsearch": query,
            "format": "json", "srlimit": 1
        }
        try:
            search_res = requests.get(search_url, params=search_params).json()
            if not search_res.get('query', {}).get('search'):
                return "No information found on Wikipedia."

            page_title = search_res['query']['search'][0]['title']

            # 2. Get the actual summary text of that page
            content_params = {
                "action": "query", "prop": "extracts", "exintro": True,
                "explaintext": True, "titles": page_title, "format": "json"
            }
            content_res = requests.get(search_url, params=content_params).json()
            pages = content_res['query']['pages']
            page_id = list(pages.keys())[0]
            return pages[page_id].get('extract', "Content unavailable.")
        except Exception as e:
            return f"Error retrieving context: {str(e)}"

    def chat(self, question):
        # 1. RETRIEVE context from Wikipedia
        context = self.fetch_wikipedia_context(question)

        # 2. AUGMENT: Create the prompt with context
        input_text = f"Context: {context}\n\nQuestion: {question}\nAnswer the question based only on the context provided."

        # 3. GENERATE answer using the Hugging Face model
        inputs = self.tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(self.device)
        outputs = self.model.generate(**inputs, max_new_tokens=150)

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- Execution ---
bot = WikiRAGBot()

while True:
    user_query = input("\n[You]: ")
    if user_query.lower() in ['exit', 'quit', 'stop']:
        break

    answer = bot.chat(user_query)
    print(f"[Bot]: {answer}")


Loading google/flan-t5-base...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running on cpu

[You]: tell me about rainbow
[Bot]: The word rainbow is a slang term for a rainbow.

[You]: tell me about michael jackson
[Bot]: Michael Jackson is an American singer and songwriter.

[You]: tell me about shreya ghoshal
[Bot]: Shreya Ghoshal is an Indian actress.
